# Fine-Tuning, Routing, and Prompt Optimization

These are different ways to improve a model system: prompts change instructions, retrieval adds current knowledge, routing chooses a model, and fine-tuning changes model behavior.


## What to learn

- Prompt changes are the fastest option for unclear instructions or formatting.
- Retrieval is appropriate for changing or private knowledge.
- Routing sends easy and hard requests to different models or workflows.
- Fine-tuning helps with repeated behavior or style that examples can teach.

## Decision rule

Build an evaluation set first. Try prompt and workflow changes before training, use retrieval for facts that change, route only when the quality-cost tradeoff is measurable, and fine-tune only with representative data and regression tests.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Cascade routing with a confidence signal, plus the cost math.

The core production routing pattern: try the cheap model, escalate when
confidence is low, and measure the cost/quality tradeoff.
"""
QUERIES = [  # (query, difficulty 0-1) — difficulty unknown to the router
    ("What's your refund window?", 0.1),
    ("Summarize this 40-page contract's indemnity risks", 0.9),
    ("Translate 'hello' to French", 0.05),
    ("Reconcile these three conflicting API specs", 0.8),
    ("What are your support hours?", 0.1),
]

PRICE = {"small": 0.4, "frontier": 8.0}  # $ per M tokens (illustrative)


# Define the data shape and small operations before running them.
def small_model(query, difficulty):
    """Cheap model: good on easy queries; confidence tracks difficulty."""
    confidence = 1.0 - difficulty
    quality = 0.95 if difficulty < 0.5 else 0.55
    return {"quality": quality, "confidence": confidence}


def cascade(query, difficulty, threshold=0.6):
    result = small_model(query, difficulty)
    if result["confidence"] >= threshold:
        return {"model": "small", "quality": result["quality"]}
    return {"model": "frontier", "quality": 0.97}  # escalate


routed = [cascade(q, d) for q, d in QUERIES]
n_small = sum(1 for r in routed if r["model"] == "small")
cost = n_small * PRICE["small"] + (len(routed) - n_small) * PRICE["frontier"]
cost_all_frontier = len(routed) * PRICE["frontier"]
avg_q = sum(r["quality"] for r in routed) / len(routed)

for (q, _), r in zip(QUERIES, routed):
    print(f"  [{r['model']:>8}] q={r['quality']:.2f}  {q[:50]}")
print(f"\ncost: ${cost:.1f} vs all-frontier ${cost_all_frontier:.1f} "
      f"({(1 - cost / cost_all_frontier):.0%} saved) at avg quality {avg_q:.2f}")
# The whole game is the confidence threshold: sweep it and you trace the
# cost/quality Pareto curve — the operating point is a product decision.


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
